# 03 - 肿瘤外科 LoRA 微调（Qwen3-4B）⭐ 云端

**ClawTeam v3.1 真协作 Harness 版**

- **基座**：Qwen3-4B-Instruct（云端 4090，约 12GB 显存）
- **数据**：真实 Benchmark（MedQA + CMExam）筛选 + LLM 改写扩充，~2000 条
- **时间**：2-3 小时
- **成本**：~¥10

**前置准备**：
```bash
# 1. 下载 benchmark
python eval/datasets/download_medqa.py
python eval/datasets/download_cmexam.py
python eval/datasets/download_medbullets.py

# 2. 准备外科训练数据
python eval/data_generators/prepare_surgeon_data.py
```

## Step 0: 修复 Windows 编码 + HF 镜像（云端不需要镜像，本地需要）

In [1]:
import os, sys, pathlib, builtins

# AutoDL 云端如果在国内，建议保留镜像；海外环境可注释
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# Windows GBK 兼容（云端 Linux 不需要，但保留无害）
_orig_read_text = pathlib.Path.read_text
def _utf8_read_text(self, encoding=None, errors=None, **kw):
    return _orig_read_text(self, encoding=encoding or 'utf-8', errors=errors, **kw)
pathlib.Path.read_text = _utf8_read_text

_orig_open = builtins.open
def _utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, **kw):
    if 'b' not in mode and encoding is None:
        encoding = 'utf-8'
    return _orig_open(file, mode, buffering, encoding, errors, **kw)
builtins.open = _utf8_open

print('环境配置完成')

环境配置完成


## Step 1: 环境检查

In [2]:
import torch, transformers, peft, trl
from pathlib import Path

print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print(f'Transformers: {transformers.__version__}')
print(f'PEFT: {peft.__version__}')
print(f'TRL: {trl.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.7.0+cu128, CUDA: True
Transformers: 5.7.0
PEFT: 0.19.1
TRL: 1.3.0
GPU: NVIDIA GeForce RTX 5090
VRAM: 33.7 GB


## Step 2: 配置（云端 Qwen3-4B）

In [3]:
# ===== 关键参数 =====
# 主推 Qwen3-4B（4090 24GB 完美）
# 如果 4090 紧张可用 Qwen3-1.7B-Instruct
MODEL_NAME = '/root/autodl-tmp/Qwen3-4B-Instruct-2507'  # AutoDL 5090 本地路径
# 备选: 'Qwen/Qwen3-1.7B-Instruct' / 'Qwen/Qwen3-8B-Instruct'

NOTEBOOK_DIR = Path.cwd()
EVAL_DIR = NOTEBOOK_DIR.parent
BACKEND_DIR = EVAL_DIR.parent
DATA_PATH = EVAL_DIR / 'datasets' / 'data' / 'training' / 'surgeon_train.jsonl'
OUTPUT_DIR = EVAL_DIR / 'models' / 'surgeon_qwen3_lora'
ROLE_PROMPT_PATH = BACKEND_DIR / 'workspace' / 'roles' / 'SURGEON.md'

# 训练超参
MAX_LENGTH = 2048
BATCH_SIZE = 4     # 4090 跑 4B 可以 2，跑 8B 改 1
GRAD_ACCUM = 4     # 实际 batch = 16
LEARNING_RATE = 5e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1

# LoRA 参数
LORA_R = 32          # Qwen3 推荐 r=16-32（比之前的 8 大）
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

USE_4BIT = False  # Qwen3-4B 不需要量化；7B/8B 可开

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'数据存在: {DATA_PATH.exists()}')
print(f'输出: {OUTPUT_DIR}')

数据存在: True
输出: /root/autodl-tmp/ClowTeam_NLP/backend/eval/models/surgeon_qwen3_lora


## Step 3: 加载数据

In [4]:
import json, random
from datasets import Dataset

random.seed(42)

records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            msgs = r.get('messages', [])
            if len(msgs) >= 3 and msgs[-1].get('role') == 'assistant':
                records.append({'messages': msgs})

random.shuffle(records)
n_val = max(40, int(len(records) * 0.05))
train_dataset = Dataset.from_list(records[n_val:])
val_dataset = Dataset.from_list(records[:n_val])
print(f'训练: {len(train_dataset)}, 验证: {len(val_dataset)}')

# 数据来源分布
from collections import Counter
sources = Counter(r.get('source', 'unknown') for r in records if 'source' in r)
print(f'数据来源分布: {sources}')

训练: 3591, 验证: 188
数据来源分布: Counter()


## Step 4: 加载基座模型 + LoRA

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    'dtype': torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    'device_map': 'auto',
    'trust_remote_code': True,
}

if USE_4BIT:
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.config.use_cache = False

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'显存占用: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157
显存占用: 8.32 GB


## Step 5: 训练（兼容新旧版 trl）

In [6]:
from trl import SFTTrainer, SFTConfig
import inspect

sft_init_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
config_kwargs = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_strategy='steps',
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=10,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to='none',
    seed=42,
)
if 'max_length' in sft_init_params:
    config_kwargs['max_length'] = MAX_LENGTH
elif 'max_seq_length' in sft_init_params:
    config_kwargs['max_seq_length'] = MAX_LENGTH
if 'packing' in sft_init_params:
    config_kwargs['packing'] = False

sft_config = SFTConfig(**config_kwargs)

trainer_init_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
trainer_kwargs = dict(
    model=model, args=sft_config,
    train_dataset=train_dataset, eval_dataset=val_dataset,
)
if 'processing_class' in trainer_init_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_init_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
print('开始训练 Qwen3-4B 外科 LoRA...')
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/3591 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/188 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


开始训练 Qwen3-4B 外科 LoRA...


Step,Training Loss,Validation Loss
50,1.023969,0.820587
100,0.415505,0.427217
150,0.384602,0.392924
200,0.366305,0.377170
250,0.360542,0.366169
300,0.354524,0.358287
350,0.332617,0.353257
400,0.340096,0.348271
450,0.324907,0.344060
500,0.328288,0.342604


TrainOutput(global_step=1125, training_loss=0.3914423393673367, metrics={'train_runtime': 7467.8596, 'train_samples_per_second': 2.404, 'train_steps_per_second': 0.151, 'total_flos': 5.21585544044074e+17, 'train_loss': 0.3914423393673367})

## Step 6: 保存

In [7]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(OUTPUT_DIR / 'training_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': MODEL_NAME,
        'role': 'surgeon',
        'lora_config': {
            'r': LORA_R, 'alpha': LORA_ALPHA,
            'target_modules': TARGET_MODULES,
        },
        'train_size': len(train_dataset),
        'val_size': len(val_dataset),
        'epochs': NUM_EPOCHS,
        'max_length': MAX_LENGTH,
    }, f, ensure_ascii=False, indent=2)

print(f'保存到: {OUTPUT_DIR.resolve()}')
for p in OUTPUT_DIR.glob('*'):
    if p.is_file():
        print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')

保存到: /root/autodl-tmp/ClowTeam_NLP/backend/eval/models/surgeon_qwen3_lora
  README.md: 0.0 MB
  adapter_model.safetensors: 264.3 MB
  adapter_config.json: 0.0 MB
  chat_template.jinja: 0.0 MB
  tokenizer_config.json: 0.0 MB
  tokenizer.json: 11.4 MB
  training_info.json: 0.0 MB


In [39]:
model = lora_model

## Step 7: 推理测试

In [40]:
model.config.use_cache = True
model.eval()

system_prompt = ROLE_PROMPT_PATH.read_text(encoding='utf-8') if ROLE_PROMPT_PATH.exists() else '你是肿瘤外科医生。'

test_qs = [
    '58 岁男性，右上肺 3cm 腺癌，T2N0M0，吸烟 30 年，FEV1 75% 预计值。可手术吗？建议什么术式？',
    '65 岁女性，胃癌 T3N2M0，幽门梗阻。外科评估：可切除性 + 术式 + 围手术期管理',
    '72 岁男性，肝细胞癌 5cm，肝硬化 Child A，AFP 升高。手术 vs 介入怎么选？',
]

for q in test_qs:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n=== 问: {q} ===\n答:\n{response}\n{"-"*80}')


=== 问: 58 岁男性，右上肺 3cm 腺癌，T2N0M0，吸烟 30 年，FEV1 75% 预计值。可手术吗？建议什么术式？ ===
答:
可切除性：该患者为 T2N0M0 肺腺癌，无淋巴结转移，肿瘤直径 3cm，位于右上肺，无肺不张或胸壁侵犯，可切除。术式：建议行右上肺叶切除+系统性淋巴结清扫术，因肿瘤位于上叶，肺叶切除可完整切除，且患者肺功能良好（FEV1 75%），全肺切除风险过高。淋巴清扫：需清扫第 2、4、7、10 组淋巴结，明确 N 状态。围手术期：术前戒烟 2-3 周，评估肺功能，术后监测呼吸功能，早期活动。
--------------------------------------------------------------------------------

=== 问: 65 岁女性，胃癌 T3N2M0，幽门梗阻。外科评估：可切除性 + 术式 + 围手术期管理 ===
答:
可切除性：幽门梗阻提示肿瘤侵犯幽门，但 T3N2M0 仍属可切除。术式：首选远端胃大部切除术（D2 根治术），因 N2 有淋巴结转移，需彻底清扫。淋巴清扫：D2 清扫包括胃周、肝总动脉、脾动脉、腹腔干周围淋巴结。围手术期：术前纠正营养不良，胃肠外营养；术中注意保护迷走神经；术后早期肠内营养，监测吻合口漏。
--------------------------------------------------------------------------------

=== 问: 72 岁男性，肝细胞癌 5cm，肝硬化 Child A，AFP 升高。手术 vs 介入怎么选？ ===
答:
可切除性：肝细胞癌 5cm，单发，肝功能 Child A，代偿肝体积足够，可切除。术式：行肝部分切除术（右半肝或左半肝），保证切缘阴性。淋巴清扫：肝癌不常规行淋巴清扫，但可切除肝门淋巴结以明确分期。围手术期：术前评估门静脉高压及肝功能，术后监测肝功能、AFP。该患者年轻（72岁）但肝硬化，需注意术后肝衰竭风险。
--------------------------------------------------------------------------------


## Step 8: Round 2 反驳能力测试（关键 — 验证真协作）

In [42]:
# 模拟 MDT 场景：外科看到病理 + 内科 + 放疗的意见后，应该有自己的反驳/修正
round2_test = '''【病例】
58 岁男性，肺腺癌，T3N2M0，EGFR 阳性。

【其他专家 Round 1 意见】
病理科：分化差，建议先新辅助治疗
肿瘤内科：建议先靶向（吉非替尼）2 个月，再评估手术
放疗科：建议手术后辅助放疗，剂量 50Gy

【你的 Round 1 意见】
可切除，建议右肺上叶切除 + 系统淋巴清扫

【Round 2 任务】
你已经看到所有其他专家的 Round 1 意见。请严格按以下 4 段结构输出（不要省略任何一项），不要输出 Round 1 格式：

## 同意
[列出你同意的他人观点 + 简要理由]

## 反对
[列出你不同意的他人观点 + 反对依据；如完全同意写"无明显分歧"]

## 修正
[基于他人意见，是否修正你的 Round 1 判断？如不修正写"坚持 Round 1 判断"]

## Round 2 最终意见
[结合上述思考，给出更新版的外科方案]'''

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': round2_test},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=600,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'\n=== Round 2 反驳测试 ===\n{response}')


=== Round 2 反驳测试 ===
## 同意
同意肿瘤内科建议的先靶向治疗，因为 EGFR 阳性患者新辅助靶向可提高病理缓解率。

## 反对
反对放疗科建议的术后辅助放疗，因为 N2 期患者术后辅助放疗获益有限，且可能增加肺功能损伤。

## 修正
坚持 Round 1 判断，但调整时机：先完成 2 个月靶向治疗，再评估手术。若新辅助后降期，可考虑直接手术；若进展，改用化疗或免疫。

## Round 2 最终意见
可切除性：新辅助靶向后重新评估，若降期则可行手术。术式：右肺上叶切除 + 系统淋巴结清扫（2R、4R、7、10R、11R、12R）。淋巴清扫：需清扫至少 15 个淋巴结。围手术期：术前靶向治疗 2 个月，术后继续靶向治疗 1 年，无需辅助放疗。


In [32]:
from transformers import AutoModelForCausalLM
import gc

# === 加载纯基座模型（无 LoRA） ===
del model
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
)

case_text = '''58 岁男性，肺腺癌，T3N2M0，EGFR 阳性。
其他专家意见：病理建议新辅助；内科建议靶向 2 个月；放疗建议术后 50Gy。
你的 Round 1：可切除，建议右肺上叶切除 + 系统淋巴清扫。'''

# 弱 prompt（旧版）
weak_prompt = case_text + "\n请按 Round 2 格式回答（同意/反对/修正/最终）。"

# 强 prompt（新版，4 段模板）
strong_prompt = case_text + '''

请严格按以下 4 段输出：
## 同意
[同意的他人观点]
## 反对
[反对的他人观点]
## 修正
[是否修正自己的判断]
## Round 2 最终意见
[更新版方案]'''

system = open(ROLE_PROMPT_PATH).read()

def run(model_, prompt):
    msgs = [{'role':'system','content':system}, {'role':'user','content':prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model_.device)
    out = model_.generate(**inputs, max_new_tokens=600, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("=== a) 基座 + 弱 prompt ===")
print(run(base_model, weak_prompt))
print("\n=== b) 基座 + 强 prompt ===")
print(run(base_model, strong_prompt))

# 重新加载 LoRA
from peft import PeftModel
del base_model
gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
lora_model = PeftModel.from_pretrained(base, OUTPUT_DIR)

print("\n=== c) LoRA + 弱 prompt ===")
print(run(lora_model, weak_prompt))
print("\n=== d) LoRA + 强 prompt ===")
print(run(lora_model, strong_prompt))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

=== a) 基座 + 弱 prompt ===
**Round 2：外科医生回应（基于MDT多学科意见）**

---

- **同意**：  
  内科建议靶向治疗2个月，**我部分同意**，但需结合肿瘤可切除性与患者整体耐受性进行调整。  
  理由：该患者为T3N2M0肺腺癌，EGFR阳性，属于可切除的局部晚期肺癌（IIIB期），且EGFR靶向治疗在局部晚期患者中已被证实可改善肿瘤负荷、提高手术可行性。然而，**靶向治疗应作为术前新辅助治疗的一部分，而非独立先行2个月**。  
  若患者在术前接受2个月靶向治疗，可能显著缩小肿瘤体积，改善可切除性，降低手术风险，尤其对于T3期肿瘤（肿瘤侵犯胸膜或邻近结构）具有潜在益处。因此，**建议将靶向治疗纳入新辅助方案，与手术同步规划**，而非“先靶向2个月再手术”——这可能延误根治性手术时机，尤其在肿瘤未充分缩小前。

- **反对**：  
  放疗建议“术后50Gy”**我明确反对**。  
  理由：该患者为T3N2M0，已明确存在区域淋巴结转移（N2），且为EGFR阳性，属于**可切除的局部晚期肺癌**。术后放疗（如50Gy）在该分期中**缺乏循证支持**，且会显著增加放射性肺炎、肺功能下降、生活质量受损等风险。  
  更重要的是，**N2淋巴结转移提示存在区域播散，若不进行术前系统性治疗（如新辅助靶向或化疗），术后放疗无法替代根治性切除的局部控制作用**。  
  因此，**不应在手术后进行放疗**，而应优先考虑术前新辅助治疗以缩小肿瘤、控制淋巴结病灶，再行根治性切除。

- **修正**：  
  基于病理建议“新辅助治疗”以及EGFR阳性特征，我**修正原建议**：  
  - 原建议：右肺上叶切除 + 系统淋巴清扫  
  - **修正为**：**右肺上叶切除 + N2站系统淋巴结清扫（包括纵隔及锁骨上区N2站）**，并**在手术前启动EGFR靶向治疗（如奥希替尼）作为新辅助治疗**，治疗周期建议为4–6周，治疗后评估肿瘤缩小情况及可切除性。  
  理由：T3N2M0患者若不进行新辅助治疗，手术切除后复发风险高，尤其在N2转移存在的情况下。EG

=== b) 基座 + 强 prompt ===
## 同意  
病理建议新辅助治疗：同意。T3N2M0 肺腺癌伴 EGFR 阳性，肿瘤侵犯邻近结构（T

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


=== c) LoRA + 弱 prompt ===
同意：病理新辅助建议合理，EGFR 阳性患者新辅助靶向可提高病理完全缓解率。
反对：反对直接手术，因 N2 淋巴结转移，直接手术复发风险高，新辅助靶向更优。
修正：坚持 Round 1 判断，但调整时机：先靶向治疗 2 个月，再评估手术。
最终：可切除，但需新辅助靶向治疗后再手术，术式同 Round 1，淋巴清扫范围扩大至 4 部分。

=== d) LoRA + 强 prompt ===
## 同意
同意病理科关于新辅助靶向治疗可提高病理完全缓解率的观点，以及内科建议的奥希替尼 2 个月治疗。

## 反对
反对放疗科建议的术后放疗，因 N2 淋巴结转移对放疗不敏感，且术后放疗增加肺纤维化风险；反对内科直接靶向 2 个月后评估，应延长至 3 个月以更好控制微转移。

## 修正
坚持 Round 1 判断：可切除，右肺上叶切除 + 系统淋巴清扫。但修正为：先新辅助奥希替尼 3 个月，再评估手术。

## Round 2 最终意见
可切除，建议新辅助奥希替尼 3 个月后行右肺上叶切除 + 系统淋巴清扫（包括 2R、4R、7、9、10、11、12 站），术后继续奥希替尼辅助治疗 36 个月。不推荐术后放疗。


In [34]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# 加载基座
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

# 叠加 LoRA
lora_model = PeftModel.from_pretrained(base, OUTPUT_DIR)

# 验证
print("LoRA adapter:", lora_model.peft_config)
print("Trainable params:", sum(p.numel() for p in lora_model.parameters() if p.requires_grad))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LoRA adapter: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='/root/autodl-tmp/Qwen3-4B-Instruct-2507', revision=None, inference_mode=True, r=32, target_modules={'v_proj', 'gate_proj', 'o_proj', 'up_proj', 'down_proj', 'q_proj', 'k_proj'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_ty

In [35]:
lora_params = sum(
    p.numel() for n, p in lora_model.named_parameters() if 'lora' in n.lower()
)
print(f"LoRA 参数: {lora_params:,}")  # 应该 ≈ 66,060,288

LoRA 参数: 66,060,288
